**Linda Zier**

**ST 554**

**Final Project**

**Goal**

For this project we :

*   added our project and files to our github repo, committing often to show our progress.
*   wrote a Jupyter notebook that fits a machine learning model using pyspark’s MLlib module. In that same notebook we wrote code to read in a stream of data (data that we produced ourselves using a .py file that is also kept in the repo).
*   we used the model to do predictions on the stream and wrote those out to the console.


**Data**

The data is modified from the UCI machine learning repository. The file power_ml_data.csv is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. The study was about relating power consumption from different zones of Tetouan city to various factors like time of day, temperature, and
humidity.


*   We used a chunk to build our model.
*   We then 'streamed data' to a folder that we monitored. As data came in we used our fitted model to predict on the incoming data.





# Fitting the Model

We created a Jupyter notebook for the model fitting part and the streaming part below. We completed the following:

*   read the data into a standard pandas data frame using the pd.read_csv() function
*   converted this to a spark data frame
*   treated the Power_Zone_3 variable as our response variable and used the other variables as predictors

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, \
                               OneHotEncoder, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.getOrCreate()

# read in data/power_ml_data.csv as pandas dataframe
powerDF=pd.read_csv("data/power_ml_data.csv")

#convert to spark dataframe
powerDF=spark.createDataFrame(powerDF)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 14:02:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 14:02:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/29 14:02:40 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/29 14:02:40 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


### Creating the Pipeline

We fit an elastic net model using CV with the steps below.
The transformations below each used an MLlib function that we put into a pipeline.

*   We used an SQL transformer to cast the hour variable as a DoubleType.

*   We binarized the Hour column based on the column being less than 6.5 or not (night vs day essentially).

*   The month column was one-hot encoded.
*   We Ran a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns. We did this by:
    - first by using a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator
    
    - then we had a PCA transformer for use in our pipeline.
    - we used two PCs in our transformation.


*   We renamed our response variable as label

*   We used VectorAssembler() to put our predictors into a features. The predictors are:

    – two fitted PCA features

    – binary Hour variable

    – Power_Zone_1

    – Power_Zone_2

    – Month indicator variables


In [2]:
# check the data types
powerDF.printSchema()

# cast the hour as double since it is a long
sql = SQLTransformer(statement = '''
                     SELECT *, 
                     CAST(Hour AS DOUBLE) AS HourD FROM __THIS__
                     ''')

# binarize night vs day
binarizer = Binarizer(threshold=6.5, inputCol="HourD", outputCol="Hour_bin")

# one hot encode month
ohe = OneHotEncoder(inputCols=["Month"], outputCols=["Month_ohe"])

#VectorAssembler to bundle features together for pca
pca_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", 
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="pca_input")

# pca with 2 components
pca = PCA(k=2, inputCol="pca_input", outputCol="pca_features")

# response variable to label
sql_label= SQLTransformer(statement = '''
                          SELECT *,
                          Power_Zone_3 AS label FROM __THIS__
                          ''')
# assemble final features
assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin", "Power_Zone_1", 
               "Power_Zone_2","Month_ohe"],
    outputCol="features")

print("TRANSFORMATIONS COMPLETE")   
                     

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

TRANSFORMATIONS COMPLETE


In [3]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sql, binarizer, ohe, pca_assembler, 
                             pca, sql_label, assembler])
fittedPipeline = pipeline.fit(powerDF)
transformedDF=fittedPipeline.transform(powerDF)

print("PIPELINE COMPLETE")

PIPELINE COMPLETE


### Fitting an Elastic Net Model
Next we used the CrossValidator() function and the LinearRegression() function to fit an elastic net model. We did multiple combinations of reg and elastic net parameters.  We fit the model using 5-fold cross validation with root mean square error (RMSE) as the criteria: we're training 5 separate models (one per fold) and averaging their RMSE's together to get their RMSE for that combination. We then report the optimal values chosen for the tuning parameters and the CV error which is the RMSE from the best model.


In [4]:

# setting up elastic net model
lr= LinearRegression(elasticNetParam=0.5)


#  grid for the regParam and elasticNetParam
paramGrid= ParamGridBuilder() \
    .addGrid(lr.regParam,[0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam,[ 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# 5-fold CV with rmse evaluator
cv=CrossValidator(estimator=lr,
                   estimatorParamMaps = paramGrid,  
                   evaluator = RegressionEvaluator(metricName= 'rmse'),
                   numFolds=5)

# fit the model
cvModel=cv.fit(transformedDF)

# Report the optimal values chosen for the tuning parameters
print("Optimal regParam:", cvModel.bestModel.getRegParam())
print("Optimal elasticNetParam:", cvModel.bestModel.getElasticNetParam())

# report RMSE errors - if you want to see all 11 x 11 = 121 of them
# print("RMSE errors= ", cvModel.avgMetrics)

# report lowest RMSE
print("CV RMSE = ", min(cvModel.avgMetrics))


26/04/29 14:02:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/29 14:02:59 WARN Instrumentation: [0e6247ac] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:03:01 WARN Instrumentation: [0e6247ac] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 14:03:04 WARN Instrumentation: [54da6aa7] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:03:05 WARN Instrumentation: [54da6aa7] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 14:03:06 WARN Instrumentation: [cbd137f0] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:03:06 WARN Instrumentation: [cbd137f0] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Optimal regParam: 0.05
Optimal elasticNetParam: 0.05
CV RMSE =  2147.7510671569535


We reported the training set RMSE using our fitted model as a transformer and
evaluating on the entire training set.

We then created a residual column (label - prediction) and printed the data frame with these
residuals.  I also printed a summary table as a sanity check.

In [5]:
# report training set RMSE by using fitted model as a transformer
predictions = cvModel.transform(transformedDF)
trainRMSE = RegressionEvaluator(metricName= 'rmse').evaluate(predictions)
print("Training RMSE=", trainRMSE)

# create residual column and display results
print("Training set residuals and summary:")
predictions = predictions.withColumn("residual", col("label") - col("prediction"))
predictions.select("label", "prediction", "residual").show()

#sanity check on residual distribution
predictions.select("label", "prediction", "residual") \
    .summary("mean", "stddev", "min", "max") \
    .show()

Training RMSE= 2147.097295248907
Training set residuals and summary:
+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20879.67067851868|-638.7068185186799|
|20131.08434| 18660.12933347464|1470.9550065253607|
|19668.43373| 18204.65111302243| 1463.782616977569|
|18899.27711| 17590.56994082658|  1308.70716917342|
|18442.40964|16997.220954537857|1445.1886854621443|
|18130.12048|16517.614600106637|1612.5058798933642|
|17945.06024|16093.185409630987|1851.8748303690118|
|17459.27711|15722.637868825768|1736.6392411742308|
|17025.54217| 15270.99557721907| 1754.546592780931|
|16794.21687|14938.301657159238|1855.9152128407623|
|16638.07229| 14652.38338677062|1985.6889032293802|
|16395.18072|14414.904162932275|1980.2765570677257|
|16117.59036| 14082.85024323325|2034.7401167667504|
| 15822.6506|13624.855902916526|2197.7946970834746|
|15672.28916|13450.339719713847| 2221.949440286

+-------+------------------+------------------+--------------------+
|summary|             label|        prediction|            residual|
+-------+------------------+------------------+--------------------+
|   mean|17831.197607816746|17831.197607816735|2.170263271911964...|
| stddev| 6622.590469869357| 6264.797048257809|   2147.120052821209|
|    min|        5935.17407|2770.0141022117996|  -9287.102805648454|
|    max|       47598.32636| 39172.36136979249|  11510.363943291923|
+-------+------------------+------------------+--------------------+



# Handling Streaming Data
We downloaded a file from: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv
and stored it in our final_project/data directory.  This was our source of random sampling.
### Reading a Stream
We read in a stream in the form of .csv files. I created a folder (using mkdir in terminal) called streaming_data where I will read in my .csv files. The schema is set to that of the original data since that is what our incoming data will look like and a header is assumed present.

In [6]:
# set the schema to that of the original data
stream_schema=powerDF.schema
print(stream_schema)

# set up readstream with a header
streamDF = spark.readStream.schema(stream_schema).option("header", True)\
           .csv("streaming_data")

#showing current working directory
#import os
#os.getcwd()

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])



### Transform/Aggregation Step

In this code block we use our model transformer to obtain predictions from the incoming data stream. 

First we created a residual column as we did in the previous section and return only label, prediction, and residual.

Then with another transformation on the (original) stream, we modified the response variable to be called label.

Lastly we joined our two transformations based on the label variable.

In [12]:
# ---Transformation 1:---

# apply transformer to the data stream
streamPredictions = cvModel.transform(fittedPipeline.transform(streamDF))

#add residual column = label - predictions
streamResiduals=streamPredictions.withColumn("residual",col("label")- col("prediction")) \
                                  .select("label","prediction", "residual")
                                              
# ---Transformation 2:---

# rename response to
streamLabeled=sql_label.transform(streamDF).select("label")

# --- Joining the 2 ---
streamJoined = streamResiduals.join(streamLabeled, on="label", how="inner")

print("TRANSFORMATION/AGGREGATION STEP COMPLETE")

TRANSFORMATION/AGGREGATION STEP COMPLETE


### Writing Step

 We used the append output  mode to write our stream to the console and started our query.

In [14]:
# Write strem to console in append mode and start query
#query = streamJoined.writeStream.outputMode("append").format("console")

query = streamJoined \
                .join(streamLabeled, "label", "inner") \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

query.awaitTermination(20)
query.stop()

26/04/29 14:20:06 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-12704056-4f20-433f-9dd8-f1b57de742bb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 14:20:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/29 14:20:27 WARN DAGScheduler: Failed to cancel job group 2b577095-159a-4928-a391-438f7172228a. Cannot find active jobs for it.
26/04/29 14:20:27 WARN DAGScheduler: Failed to cancel job group 2b577095-159a-4928-a391-438f7172228a. Cannot find active jobs for it.
